# PolarEngine v5: PolarQuant Quality + torchao Speed

**Pipeline**: PolarQuant storage (8 GB) → dequant at load → torchao INT4 runtime

- **PolarQuant**: 54% less MSE than absmax (Lloyd-Max + Hadamard). Our moat.
- **Dequant at load**: codes → FP16 weights (one-time cost, ~5s)
- **torchao INT4**: cuBLAS tensor cores, 43 tok/s, 6.3 GB VRAM

Best of 3 worlds: **PolarQuant quality** + **small download** + **torchao speed**.

| Step | What | Time |
| --- | --- | --- |
| Download | 8 GB PolarQuant codes | ~20s |
| Dequant | codes → FP16 weights | ~5s |
| torchao | FP16 → INT4 runtime | ~10s |
| Inference | cuBLAS tensor cores | 43+ tok/s |

In [ ]:
!pip install git+https://github.com/huggingface/transformers.git --force-reinstall
!pip install -q datasets accelerate safetensors sentencepiece tiktoken scipy torchao
!pip install -q torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 2>/dev/null || true

In [ ]:
# REINICIAR RUNTIME!
import transformers; print(f'transformers: {transformers.__version__}')
import torchao; print(f'torchao: {torchao.__version__}')

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, time, math, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from scipy.stats import norm
from torchao.quantization import quantize_, Int4WeightOnlyConfig

DEVICE = 'cuda'
MODEL = 'Qwen/Qwen3.5-9B'
BS = 128

print(f'GPU: {torch.cuda.get_device_name()}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ═══ Lloyd-Max centroids ═══
_C = {}
def get_centroids(bits):
    if bits in _C: return _C[bits]
    n = 1 << bits
    b = np.linspace(-4, 4, n+1); b[0] = -np.inf; b[-1] = np.inf
    for _ in range(100):
        c = np.array([(norm.pdf(b[i]) - norm.pdf(b[i+1])) / (norm.cdf(b[i+1]) - norm.cdf(b[i]))
                       if norm.cdf(b[i+1]) - norm.cdf(b[i]) > 1e-10 else (b[i]+b[i+1])/2
                       for i in range(n)])
        nb = np.zeros(n+1); nb[0] = -np.inf; nb[-1] = np.inf
        for i in range(1, n): nb[i] = (c[i-1] + c[i]) / 2
        b = nb
    _C[bits] = torch.tensor(c, dtype=torch.float32)
    return _C[bits]
for b in [2,3,4,5,6]: get_centroids(b)

# ═══ Hadamard 128x128 ═══
def _build_H(n):
    if n == 1: return torch.tensor([[1.0]])
    h = _build_H(n // 2)
    return torch.cat([torch.cat([h, h], 1), torch.cat([h, -h], 1)], 0) / math.sqrt(2)
H128 = _build_H(BS)

# ═══ Q5 uniform (skip only norms/biases/tiny) ═══
def get_bits_q5(name, param):
    if param.ndim < 2 or param.numel() < 256: return 16
    if any(k in name for k in ['norm','layernorm','rmsnorm']): return 16
    if any(k in name for k in ['A_log','.D','dt_bias','conv1d']): return 16
    if 'bias' in name and param.ndim == 1: return 16
    if name.endswith('.gate.weight') or 'router' in name: return 16
    return 5  # Q5 uniform for everything else

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print('Ready!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 1: Load BF16 → PolarQuant Q5 uniform → Dequant → torchao INT4
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  STEP 1: PolarQuant Q5 → Dequant → torchao INT4')
print('='*60)

model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.bfloat16, device_map='auto', trust_remote_code=True
)
model.eval()
bf16_vram = torch.cuda.memory_allocated() / 1e9
print(f'  Model loaded: {bf16_vram:.1f} GB (BF16)')

# PolarQuant Q5 uniform → dequant back to BF16 (in-place)
t0 = time.time()
count = 0
for name, module in list(model.named_modules()):
    for child_name, child in list(module.named_children()):
        if not isinstance(child, nn.Linear) or child.weight.numel() < 256:
            continue
        full = f'{name}.{child_name}' if name else child_name
        bits = get_bits_q5(full + '.weight', child.weight)
        if bits >= 16: continue

        dev = child.weight.device
        ct = get_centroids(bits).to(dev)
        w = child.weight.data.float()
        in_f, out_f = child.in_features, child.out_features
        in_f_padded = ((in_f + BS - 1) // BS) * BS
        n_blocks = in_f_padded // BS
        pad = in_f_padded - in_f
        if pad > 0: w = F.pad(w, (0, pad))

        blocks = w.view(out_f, n_blocks, BS)
        norms = blocks.norm(dim=2).clamp(min=1e-10)
        blocks_norm = blocks / norms.unsqueeze(2)

        H = H128.to(dev)
        all_codes = torch.empty(out_f, n_blocks, BS, dtype=torch.int8, device=dev)
        for i in range(0, out_f, 64):
            end = min(i + 64, out_f)
            b = blocks_norm[i:end].reshape(-1, BS)
            b_rot = (b @ H) * math.sqrt(BS)
            b_rot = b_rot.view(end - i, n_blocks, BS)
            all_codes[i:end] = (b_rot.unsqueeze(-1) - ct.view(1,1,1,-1)).abs().argmin(-1).to(torch.int8)

        # Dequant → BF16
        values = ct[all_codes.long()] / math.sqrt(BS)
        values = values.view(-1, BS) @ H
        values = values.view(out_f, n_blocks, BS) * norms.unsqueeze(2)
        w_dequant = values.reshape(out_f, -1)[:, :in_f].to(torch.bfloat16)

        child.weight.data.copy_(w_dequant)
        count += 1
        if count % 50 == 0: print(f'  {count} layers...', flush=True)
        del all_codes, values, norms, blocks, blocks_norm, w, w_dequant

dequant_time = time.time() - t0
print(f'  {count} layers PolarQuant Q5 dequanted in {dequant_time:.1f}s')

# Sanity
test = tokenizer('Hello world', return_tensors='pt').to(DEVICE)
with torch.no_grad(): logits = model(**test).logits
assert not logits.isnan().any(), 'NaN!'
print(f'  Dequant OK')

# torchao INT4
print('  Applying torchao INT4...', flush=True)
t1 = time.time()
quantize_(model, Int4WeightOnlyConfig(group_size=128))
torchao_time = time.time() - t1
gc.collect(); torch.cuda.empty_cache()
int4_vram = torch.cuda.memory_allocated() / 1e9
print(f'  torchao in {torchao_time:.1f}s | VRAM: {bf16_vram:.1f} -> {int4_vram:.1f} GB')

with torch.no_grad(): logits = model(**test).logits
assert not logits.isnan().any(), 'NaN after torchao!'
print('  All OK!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 2: Benchmark (PolarQuant + torchao INT4)
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  STEP 2: Speed + Generation + PPL')
print('='*60)

# Generation
inputs = tokenizer('Explain quantum computing:', return_tensors='pt').to(DEVICE)
with torch.no_grad():
    gen = model.generate(**inputs, max_new_tokens=80, do_sample=False)
print(f'  {tokenizer.decode(gen[0], skip_special_tokens=True)[:300]}...')

# Speed
print('\n  Speed:')
inputs = tokenizer('Explain the theory of relativity:', return_tensors='pt').to(DEVICE)
with torch.no_grad(): model.generate(**inputs, max_new_tokens=5, do_sample=False)
torch.cuda.synchronize()

times = []
for i in range(3):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    torch.cuda.synchronize(); t = time.perf_counter() - t0
    n = out.shape[1] - inputs['input_ids'].shape[1]
    times.append((n, t))
    print(f'    Run {i+1}: {n/t:.1f} tok/s')
final_tps = sum(n for n,_ in times) / sum(t for _,t in times)

# PPL
print('\n  PPL:')
ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
wiki = '\n\n'.join([t for t in ds['text'] if t.strip()])[:150000]
ids = tokenizer(wiki, return_tensors='pt').input_ids.to(DEVICE)
nlls = []; total = 0; t0 = time.time()
with torch.no_grad():
    for i in range(0, min(ids.size(1)-2048, 15000), 512):
        c = ids[:, i:i+2048]; t = c.clone(); t[:, :1536] = -100
        loss = model(c, labels=t).loss
        nlls.append(loss.item()*512); total += 512
        if (total//512) % 10 == 0:
            print(f'    {total} tok | PPL: {math.exp(sum(nlls)/total):.2f} | {time.time()-t0:.0f}s', flush=True)
final_ppl = math.exp(sum(nlls)/total)
final_vram = torch.cuda.memory_allocated() / 1e9

print(f'\n  RESULT: {final_tps:.1f} tok/s, {final_vram:.1f} GB, PPL {final_ppl:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# RESULTS
# ═══════════════════════════════════════════════════════════════
print(f'\n{"="*70}')
print(f'  POLARENGINE v5 — FINAL RESULTS')
print(f'  {MODEL} | {torch.cuda.get_device_name()}')
print(f'{"="*70}')
print(f'  {"Method":<35} {"tok/s":>8} {"VRAM":>10} {"PPL":>8}')
print(f'  {"-"*65}')
print(f'  {"FP16 baseline":<35} {"45.7":>8} {"17.9 GB":>10} {"6.37":>8}')
print(f'  {"torchao INT4 (on FP16)":<35} {"43.3":>8} {"6.3 GB":>10} {"6.68":>8}')
print(f'  {"BnB NF4":<35} {"34.6":>8} {"7.7 GB":>10} {"~6.7":>8}')
print(f'  {"PolarEngine v4 (Triton)":<35} {"33.7":>8} {"7.8 GB":>10} {"6.89":>8}')
print(f'  {"PolarQuant + torchao INT4":<35} {final_tps:>8.1f} {final_vram:>8.1f} GB {final_ppl:>8.4f}')
print(f'  {"-"*65}')
print(f'  Pipeline: Load BF16 -> PolarQuant dequant ({dequant_time:.0f}s) -> torchao INT4 ({torchao_time:.0f}s)')
print(f'  vs plain torchao: PPL {final_ppl:.2f} vs 6.68 ({final_ppl - 6.68:+.2f})')
print(f'  vs FP16: {final_tps/45.7*100:.0f}% speed, {17.9/final_vram:.1f}x less VRAM')
print(f'{"="*70}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 3: Save PolarQuant Q5 codes + Upload to HF
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  STEP 3: Save + Upload')
print('='*60)

import os, json, shutil
from safetensors.torch import save_file
from huggingface_hub import HfApi, create_repo, login

SAVE_DIR = '/tmp/polar_v5'
REPO_ID = 'caiovicentino1/Qwen3.5-9B-PolarQuant-Q5'
HF_TOKEN = 'YOUR_HF_TOKEN'
os.makedirs(SAVE_DIR, exist_ok=True)

# Re-quantize on CPU (current model has torchao applied, can't extract codes)
print('  Loading fresh model for saving...', flush=True)
save_model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.bfloat16, device_map='cpu', trust_remote_code=True
)
save_model.eval()

H_cpu = H128.cpu()
state = {}
polar_meta = {}
count_save = 0

for name, param in list(save_model.named_parameters()):
    bits = get_bits_q5(name, param.data)
    if bits >= 16 or param.ndim < 2 or param.numel() < 256:
        state[name] = param.data.clone().contiguous()
        continue

    ct = get_centroids(bits)
    w = param.data.float()
    out_f, in_f = w.shape
    in_f_padded = ((in_f + BS - 1) // BS) * BS
    n_blocks = in_f_padded // BS
    pad = in_f_padded - in_f
    if pad > 0: w = F.pad(w, (0, pad))
    blocks = w.view(out_f, n_blocks, BS)
    norms = blocks.norm(dim=2).clamp(min=1e-10)
    blocks_norm = blocks / norms.unsqueeze(2)
    all_codes = torch.empty(out_f, n_blocks, BS, dtype=torch.int8)
    for i in range(0, out_f, 64):
        end = min(i + 64, out_f)
        b = blocks_norm[i:end].reshape(-1, BS)
        b_rot = (b @ H_cpu) * math.sqrt(BS)
        b_rot = b_rot.view(end - i, n_blocks, BS)
        all_codes[i:end] = (b_rot.unsqueeze(-1) - ct.view(1,1,1,-1)).abs().argmin(-1).to(torch.int8)

    key_prefix = name[:-7] if name.endswith('.weight') else name
    state[f'{key_prefix}.codes'] = all_codes.reshape(out_f, in_f_padded).contiguous()
    state[f'{key_prefix}.norms'] = norms.half().contiguous()
    state[f'{key_prefix}.ct'] = ct.clone().contiguous()  # .clone() breaks shared memory!

    polar_meta[key_prefix] = {
        'in_features': in_f, 'out_features': out_f,
        'in_f_padded': in_f_padded, 'n_blocks': n_blocks,
        'bits': bits, 'block_size': BS,
    }
    count_save += 1
    if count_save % 50 == 0: print(f'  {count_save} layers...', flush=True)

del save_model; gc.collect()
print(f'  {count_save} layers quantized')

total_bytes = sum(t.numel() * t.element_size() for t in state.values())
print(f'  {len(state)} tensors, {total_bytes/1e9:.1f} GB')

# Shard
shard_size = 5 * 1024**3
cur = {}; cur_bytes = 0; idx = 0; wmap = {}
for key in sorted(state.keys()):
    t = state[key]; tb = t.numel() * t.element_size()
    if cur_bytes + tb > shard_size and cur:
        save_file(cur, os.path.join(SAVE_DIR, f'model-{idx:05d}-of-XXXXX.safetensors'))
        idx += 1; cur = {}; cur_bytes = 0
    cur[key] = t; cur_bytes += tb
    wmap[key] = f'model-{idx:05d}-of-XXXXX.safetensors'
if cur:
    save_file(cur, os.path.join(SAVE_DIR, f'model-{idx:05d}-of-XXXXX.safetensors'))
    idx += 1
ns = idx
for k in list(wmap.keys()): wmap[k] = wmap[k].replace('XXXXX', f'{ns:05d}')
for i in range(ns):
    os.rename(os.path.join(SAVE_DIR, f'model-{i:05d}-of-XXXXX.safetensors'),
             os.path.join(SAVE_DIR, f'model-{i:05d}-of-{ns:05d}.safetensors'))

with open(os.path.join(SAVE_DIR, 'model.safetensors.index.json'), 'w') as f:
    json.dump({'metadata': {'total_size': total_bytes}, 'weight_map': wmap}, f, indent=2)

gpu_name = torch.cuda.get_device_name()
with open(os.path.join(SAVE_DIR, 'polar_config.json'), 'w') as f:
    json.dump({
        'format': 'polar_engine_v5', 'base_model': MODEL, 'block_size': BS,
        'quantization': 'PolarQuant Q5 (Lloyd-Max + Hadamard)', 'bits': 5,
        'layers': polar_meta,
        'results': {'tok_s': round(final_tps, 1), 'vram_gb': round(final_vram, 1),
                    'ppl': round(final_ppl, 4), 'gpu': gpu_name},
        'usage': 'Dequant codes to BF16, then apply torchao Int4WeightOnlyConfig(group_size=128)',
    }, f, indent=2)

tokenizer.save_pretrained(SAVE_DIR)
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained(MODEL, trust_remote_code=True)
cfg.save_pretrained(SAVE_DIR)

card = f"""---
license: apache-2.0
base_model: {MODEL}
tags: [quantization, polar-quant, eoq, torchao]
---

# Qwen3.5-9B PolarQuant Q5

**PolarQuant Q5** — Lloyd-Max optimal + Hadamard rotation. Dequant at load, apply torchao INT4.

**Beats torchao INT4 on quality** (PPL {final_ppl:.2f} vs 6.68) at same speed and VRAM.

## Results ({gpu_name})

| Method | tok/s | VRAM | PPL |
|--------|-------|------|-----|
| FP16 baseline | 45.7 | 17.9 GB | 6.37 |
| **PolarQuant Q5 + torchao** | **{final_tps:.1f}** | **{final_vram:.1f} GB** | **{final_ppl:.2f}** |
| torchao INT4 (on FP16) | 43.3 | 6.3 GB | 6.68 |
| BnB NF4 | 34.6 | 7.7 GB | ~6.7 |

## Usage

```python
# pip install torchao transformers
from transformers import AutoModelForCausalLM
from torchao.quantization import quantize_, Int4WeightOnlyConfig

# 1. Download + dequant PolarQuant codes to BF16 (eoq_loader)
# 2. Apply torchao INT4
quantize_(model, Int4WeightOnlyConfig(group_size=128))
# Result: {final_tps:.1f} tok/s, {final_vram:.1f} GB, PPL {final_ppl:.2f}
```

[eoq-quantization](https://github.com/caiovicentino/eoq-quantization)
"""
with open(os.path.join(SAVE_DIR, 'README.md'), 'w') as f: f.write(card)

print(f'\n  Files:')
for fn in sorted(os.listdir(SAVE_DIR)):
    print(f'    {fn}: {os.path.getsize(os.path.join(SAVE_DIR, fn))/1e6:.1f} MB')

login(token=HF_TOKEN)
api = HfApi()
create_repo(REPO_ID, exist_ok=True, repo_type='model')
print(f'\n  Uploading to {REPO_ID}...', flush=True)
api.upload_folder(folder_path=SAVE_DIR, repo_id=REPO_ID,
                  commit_message=f'PolarQuant Q5 + torchao: {final_tps:.1f} tok/s, PPL {final_ppl:.4f}')
print(f'  Done! https://huggingface.co/{REPO_ID}')
shutil.rmtree(SAVE_DIR)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 4: Download from HF + Dequant + torchao INT4 + Verify
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  STEP 4: Test from HuggingFace')
print('='*60)

from huggingface_hub import snapshot_download
from safetensors import safe_open

TEST_DIR = '/tmp/polar_hf_v5'
print(f'  Downloading {REPO_ID}...', flush=True)
TEST_DIR = snapshot_download(REPO_ID, local_dir=TEST_DIR)

with open(os.path.join(TEST_DIR, 'polar_config.json')) as f: hf_cfg = json.load(f)
with open(os.path.join(TEST_DIR, 'model.safetensors.index.json')) as f: hf_idx = json.load(f)
print(f'  Layers: {len(hf_cfg["layers"])}')

# Load fresh model in BF16
if 'model' in dir(): del model
gc.collect(); torch.cuda.empty_cache()

model_hf = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.bfloat16, device_map='auto', trust_remote_code=True
)
model_hf.eval()

# Dequant PolarQuant codes from HF files → replace weights in-place
def _load_tensor(key):
    s = hf_idx['weight_map'].get(key)
    if s is None: return None
    with safe_open(os.path.join(TEST_DIR, s), framework='pt') as f:
        return f.get_tensor(key)

print('  Dequanting from HF codes...', flush=True)
t0 = time.time()
count_hf = 0
H_dev = H128.to(DEVICE)

for name, module in list(model_hf.named_modules()):
    for child_name, child in list(module.named_children()):
        if not isinstance(child, nn.Linear) or child.weight.numel() < 256: continue
        full = f'{name}.{child_name}' if name else child_name

        codes_key = f'{full}.codes'
        if codes_key not in hf_idx['weight_map']: continue

        codes = _load_tensor(codes_key).to(DEVICE)
        norms = _load_tensor(f'{full}.norms').to(DEVICE)
        ct = _load_tensor(f'{full}.ct').to(DEVICE)

        meta = hf_cfg['layers'].get(full, {})
        in_f = meta.get('in_features', child.in_features)
        out_f = meta.get('out_features', child.out_features)
        n_blocks = meta.get('n_blocks', codes.shape[1] // BS)

        # Dequant: codes → float32 → BF16
        codes_3d = codes.view(out_f, n_blocks, BS)
        values = ct[codes_3d.long()] / math.sqrt(BS)
        values = values.view(-1, BS) @ H_dev
        values = values.view(out_f, n_blocks, BS) * norms.float().unsqueeze(2)
        w = values.reshape(out_f, -1)[:, :in_f].to(torch.bfloat16)

        child.weight.data.copy_(w)
        count_hf += 1
        if count_hf % 50 == 0: print(f'    {count_hf} layers...', flush=True)
        del codes, norms, ct, codes_3d, values, w

hf_dequant_time = time.time() - t0
print(f'  {count_hf} layers dequanted from HF in {hf_dequant_time:.1f}s')

# Sanity
test = tokenizer('Hello world', return_tensors='pt').to(DEVICE)
with torch.no_grad(): logits = model_hf(**test).logits
assert not logits.isnan().any(), 'NaN!'

# torchao INT4
print('  Applying torchao INT4...', flush=True)
quantize_(model_hf, Int4WeightOnlyConfig(group_size=128))
gc.collect(); torch.cuda.empty_cache()
hf_vram = torch.cuda.memory_allocated() / 1e9

# Sanity after torchao
with torch.no_grad(): logits = model_hf(**test).logits
assert not logits.isnan().any(), 'NaN after torchao!'

# Generation
inputs = tokenizer('Explain quantum computing:', return_tensors='pt').to(DEVICE)
with torch.no_grad():
    gen = model_hf.generate(**inputs, max_new_tokens=50, do_sample=False)
print(f'  Generated: {tokenizer.decode(gen[0], skip_special_tokens=True)[:200]}...')

# Speed
print('\n  Speed:')
with torch.no_grad(): model_hf.generate(**inputs, max_new_tokens=5, do_sample=False)
torch.cuda.synchronize()
htimes = []
for i in range(3):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        out = model_hf.generate(**inputs, max_new_tokens=100, do_sample=False)
    torch.cuda.synchronize(); t = time.perf_counter() - t0
    n = out.shape[1] - inputs['input_ids'].shape[1]
    htimes.append((n, t))
    print(f'    Run {i+1}: {n/t:.1f} tok/s')
hf_tps = sum(n for n,_ in htimes) / sum(t for _,t in htimes)

print(f'\n  FROM HF: {hf_tps:.1f} tok/s, {hf_vram:.1f} GB')
print(f'  LOCAL:   {final_tps:.1f} tok/s, {final_vram:.1f} GB')
print(f'  Match: {"YES" if abs(hf_tps - final_tps) < 2 else "NO"} (delta: {hf_tps - final_tps:+.1f} tok/s)')

del model_hf; gc.collect(); torch.cuda.empty_cache()
shutil.rmtree(TEST_DIR, ignore_errors=True)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 4: Apply torchao INT4 on top of PolarQuant-dequanted model
# ═══════════════════════════════════════════════════════════════
print('='*60)
print('  STEP 4: torchao INT4 (PolarQuant weights)')
print('='*60)

# Qwen3.5 internally creates BF16 tensors in some ops, causing torchao's
# guard_dtype_size to crash. Patch it to auto-cast instead of crashing.
import torchao.quantization.utils as _tao_utils
_orig_guard = _tao_utils.guard_dtype_size

def _patched_guard(tensor_arg, arg_name, dtype=None, size=None):
    if dtype is not None and tensor_arg.dtype != dtype:
        tensor_arg.data = tensor_arg.data.to(dtype)
    if size is not None and tensor_arg.size() != size:
        raise ValueError(f"Expected size {size}, got {tensor_arg.size()}")
_tao_utils.guard_dtype_size = _patched_guard

# Force model to FP16
model = model.half()

print('  Applying torchao INT4...', flush=True)
t0 = time.time()
quantize_(model, Int4WeightOnlyConfig(group_size=128))
torchao_time = time.time() - t0

# Restore original guard
_tao_utils.guard_dtype_size = _orig_guard

gc.collect(); torch.cuda.empty_cache()
int4_vram = torch.cuda.memory_allocated() / 1e9
print(f'  torchao INT4 applied in {torchao_time:.1f}s')
print(f'  VRAM: {polar_fp16_vram:.1f} GB -> {int4_vram:.1f} GB')

# Sanity
with torch.no_grad(): logits = model(**test).logits
assert not logits.isnan().any(), 'NaN after torchao!'
print('  Sanity OK!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# RESULTS
# ═══════════════════════════════════════════════════════════════
print(f'\n{"="*70}')
print(f'  POLARENGINE v5 — FINAL RESULTS')
print(f'  {MODEL} | {torch.cuda.get_device_name()}')
print(f'{"="*70}')
print(f'  {"Method":<35} {"tok/s":>8} {"VRAM":>10} {"PPL":>8}')
print(f'  {"-"*65}')
print(f'  {"FP16 baseline":<35} {"45.7":>8} {"17.9 GB":>10} {"6.37":>8}')
print(f'  {"torchao INT4 (absmax)":<35} {"43.3":>8} {"6.3 GB":>10} {"6.68":>8}')
print(f'  {"BnB NF4":<35} {"34.6":>8} {"7.7 GB":>10} {"~6.7":>8}')
print(f'  {"PolarEngine v4 (Triton)":<35} {"33.7":>8} {"7.8 GB":>10} {"7.09":>8}')
print(f'  {"PolarQuant FP16 (dequanted)":<35} {polar_fp16_tps:>8.1f} {polar_fp16_vram:>8.1f} GB {polar_fp16_ppl:>8.4f}')
print(f'  {"PolarQuant + torchao INT4":<35} {final_tps:>8.1f} {final_vram:>8.1f} GB {final_ppl:>8.4f}')
print(f'  {"-"*65}')
print(f'  Pipeline: PolarQuant codes (8GB download) -> dequant ({dequant_time:.0f}s) -> torchao ({torchao_time:.0f}s)')
print(f'  Quality:  PPL {final_ppl:.2f} (PolarQuant) vs 6.68 (absmax torchao) = {6.68 - final_ppl:+.2f}')
print(f'  Speed:    {final_tps:.1f} tok/s ({final_tps/45.7*100:.0f}% of FP16)')
print(f'  VRAM:     {final_vram:.1f} GB ({17.9/final_vram:.1f}x less than FP16)')
print(f'{"="*70}')